In [0]:
%sql
use catalog dev;
use schema harika;

In [0]:
CREATE TABLE IF NOT EXISTS Emp (id INT, name STRING, salary DOUBLE);

In [0]:
DESCRIBE TABLE EXTENDED Emp;

In [0]:
DESCRIBE DETAIL Emp;

In [0]:
DESCRIBE HISTORY Emp;

In [0]:
-- Inserting some records
INSERT INTO Emp VALUES (1, 'Harika', 30000), (2, 'Bindu', 45000), (3, 'Ritesh', 50000);

In [0]:
DESCRIBE HISTORY Emp;

In [0]:
DESCRIBE DETAIL Emp;

In [0]:
select * from emp;

In [0]:
INSERT INTO dev.naval.emp VALUES (88, 'Harika')

In [0]:
describe history dev.naval.emp;

In [0]:
-- Time travel
-- We can query the older versions of the tables using version numbers or timestamps
SELECT * from dev.naval.emp version as of 5;

In [0]:
select * from dev.naval.emp timestamp as of '2026-04-08T04:44:14.000+00:00';

In [0]:
CREATE TABLE IF NOT EXISTS icebery_table (id int, name string, age int) using iceberg;

In [0]:
describe extended icebery_table;

### Incremental Data Processing

In [0]:
%python
# Use append in writing mode to add new records to the existing data.
data = [(1,'Harry'),(2,'Hermione'),(3,'Ron')]
schema = "id int, name string"
df = spark.createDataFrame(data, schema)
df.write.mode("overwrite").saveAsTable("students")

In [0]:
%python
data = [(4,'John'),(5,'Shally')]
schema = "id int, name string"
df = spark.createDataFrame(data, schema)
df.write.mode("append").saveAsTable("students")

In [0]:
%python
data = [(6, 'Prash',12)]
schema = "id int, name string, age int"
df = spark.createDataFrame(data, schema)
df.write.mode("append").option("mergeSchema","True").saveAsTable("students")

In [0]:
select * from students;

In [0]:
-- We can restore the table to the older versions if it is deleted accidentally.
DELETE FROM dev.harika.emp;

In [0]:
RESTORE TABLE dev.harika.emp TO VERSION AS OF 1;
-- We can also restore the table to the older versions if it is deleted accidentally.

In [0]:
select * from dev.harika.emp;

In [0]:
drop table dev.harika.emp;

desc history dev.harika.emp;

In [0]:
undrop table dev.harika.emp; --default retention period is 7 days or 168 hours

### Views

In [0]:
table dev.naval.driver_metadata

In [0]:
CREATE OR REPLACE VIEW driver_nationality_counts 
AS 
(
  SELECT nationality, count(*) counts
  FROM dev.harika.drivers_metadata
  GROUP BY nationality
  ORDER BY counts DESC 
)

In [0]:
SELECT * FROM driver_nationality_counts

### Temporary Views

Temp view stores the result of the query only for the session duration. Also, do not use 3 level naming for the temp views

In [0]:
CREATE TEMPORARY VIEW temp_view
AS
select c_mktsegment, count(*) as count from samples.tpch.customer group by c_mktsegment

In [0]:
USE CATALOG dev;
SHOW VIEWS IN dev.harika

### User Defined Functions

In [0]:
USE CATALOG dev;
USE SCHEMA harika

In [0]:
CREATE OR REPLACE FUNCTION create_full_name(first_name STRING, last_name STRING)
RETURNS STRING
RETURN CONCAT(first_name, ' ', last_name)

In [0]:
SELECT forename, surname, create_full_name(forename, surname) as full_name FROM dev.harika.drivers_metadata

In [0]:
-- Create another user defined function to mask the email address.
CREATE OR REPLACE FUNCTION mask_email(email STRING)
RETURNS STRING
RETURN CONCAT(repeat('*', len(email)-3), SUBSTR(email, LEN(email)-3));

In [0]:
select mask_email('abc@gmail.com');

### Task

In [0]:
-- Creating the table
USE CATALOG dev;
USE SCHEMA harika;

CREATE TABLE IF NOT EXISTS finance_data AS 
SELECT * FROM read_files('/Volumes/dev/naval/raw/task/financial_dataset.csv', format=>'csv');

SELECT * FROM finance_data

In [0]:
-- Creating a View
CREATE OR REPLACE VIEW transactions_per_branch
AS 
(
  SELECT Branch, COUNT(*) num_trans FROM finance_data 
  GROUP BY Branch
  ORDER BY num_trans DESC
);

SELECT * FROM transactions_per_branch;

In [0]:
-- CREATE USER DEFINED FUNCTION
CREATE OR REPLACE FUNCTION compute_rupees(amount DOUBLE)
RETURNS DOUBLE
RETURN ROUND((amount * 92.62),3);

SELECT AmountUSD, compute_rupees(AmountUSD) AmountRupees FROM finance_data;